# Computing Equilibria in Two-Player Normal-Form Games

*A worked walkthrough of equilibrium computation. The main thread follows Shoham & Leyton-Brown (2009, Ch. 3–4); each section also points to the original papers and to other treatments of the same idea. Full references are at the end.*

Defining a Nash equilibrium takes one sentence. Finding one takes an algorithm, and that is what this notebook is about. We will build a small solver library from scratch, starting from a single matrix product and ending with support enumeration, an algorithm that finds every equilibrium of a (nondegenerate) bimatrix game.

Each section follows the same pattern: a short piece of theory, the implementation, a demonstration on a familiar game, and a few sanity checks. Some sections end with a **Going further** note: a paper or an extension that is worth reading or implementing next. Please run the cells in order. Later functions call earlier ones.

| § | Topic | Function | Shoham & Leyton-Brown |
|---|---|---|---|
| 1 | Representation | `show_game`, `pure` | §3.2 |
| 2 | Expected payoffs | `expected_payoff` | §3.2.4 |
| 3 | Best responses | `best_response` | §3.3.2 |
| 4 | Verifying an equilibrium | `is_nash` | §3.3.2 |
| 5 | Pure-strategy equilibria | `pure_nash` | §3.3.3 |
| 6 | Iterated dominance | `eliminate_dominated` | §3.4.3, §4.5.1, §4.5.3 |
| 7 | Zero-sum games as LPs | `solve_zero_sum` | §3.4.1, §4.1 |
| 8 | Support enumeration | `solve_indifference`, `support_enumeration` | §4.2.3 |
| 9 | Domination by mixed strategies | `dominated_by_mixed` | §4.5.2 |
| 10 | Discussion | | |

In [1]:
# %pip install numpy scipy pandas nashpy
import itertools
from math import comb

import numpy as np
import pandas as pd
from scipy.optimize import linprog
import nashpy as nash   # used only as an independent reference in the checks

TOL = 1e-8
np.set_printoptions(precision=4, suppress=True)

## 1 · Representation

A finite two-player game in normal form is fully described by two matrices of the same shape:

- $A_{ij}$ is the payoff to the **row player** when she plays row $i$ and the column player plays column $j$;
- $B_{ij}$ is the payoff to the **column player** in the same situation.

Textbooks usually merge them into a single table whose cells read "row payoff, column payoff". In code it is more convenient to keep them apart, because almost every computation looks at one player's payoffs at a time.

A mixed strategy is a probability vector: $x \in \Delta^m$ for the row player, $y \in \Delta^n$ for the column player. A pure strategy is the special case that puts all its mass on one entry, which is what `pure(k, n)` builds.

One trick is worth remembering from the start. The column player choosing among columns of $B$ is the same as a row player choosing among rows of $B^\top$. Thanks to this, most functions below are written once, for the row player, and reused for the column player by passing `B.T`.

All indices are zero-based, so "strategy 2" in the book is `1` in the code.

The normal form goes back to von Neumann & Morgenstern (1944). For a more economics-oriented introduction to the same material, Osborne (2004, Ch. 2–4) is gentle and full of examples; Osborne & Rubinstein (1994) is the rigorous version.

In [2]:
GAMES = {
    "prisoners_dilemma":   (np.array([[3, 0], [5, 1]]),
                            np.array([[3, 5], [0, 1]])),
    "matching_pennies":    (np.array([[1, -1], [-1, 1]]),
                            np.array([[-1, 1], [1, -1]])),
    "rock_paper_scissors": (np.array([[0, -1, 1], [1, 0, -1], [-1, 1, 0]]),
                            np.array([[0, 1, -1], [-1, 0, 1], [1, -1, 0]])),
    "bach_or_stravinsky": (np.array([[2, 0], [0, 1]]),
                            np.array([[1, 0], [0, 2]])),
    "coordination":        (np.array([[2, 0], [0, 1]]),
                            np.array([[2, 0], [0, 1]])),
    "dominance_solvable":  (np.array([[4, 5, 6], [2, 8, 3], [3, 9, 2]]),
                            np.array([[3, 1, 2], [1, 4, 6], [0, 6, 8]])),
}

LABELS = {
    "prisoners_dilemma":   (["Cooperate", "Defect"], ["Cooperate", "Defect"]),
    "matching_pennies":    (["Heads", "Tails"], ["Heads", "Tails"]),
    "rock_paper_scissors": (["Rock", "Paper", "Scissors"], ["Rock", "Paper", "Scissors"]),
    "bach_or_stravinsky": (["Bach", "Stravinsky"], ["Bach", "Stravinsky"]),
    "coordination":        (["Left", "Right"], ["Left", "Right"]),
    "dominance_solvable":  (["U", "M", "D"], ["L", "C", "R"]),
}


def show_game(name):
    '''Display a game from GAMES as a "row payoff, column payoff" table.

    Parameters
    ----------
    name : str
        Key of the game in GAMES, e.g. "prisoners_dilemma".
    '''
    A, B = GAMES[name]
    rows, cols = LABELS[name]
    cells = [[f"{A[i, j]:g}, {B[i, j]:g}" for j in range(A.shape[1])] for i in range(A.shape[0])]
    return pd.DataFrame(cells, index=rows, columns=cols)


def pure(k, n):
    '''Write the pure strategy "always play k" as a probability vector.

    Parameters
    ----------
    k : int
        Which strategy is played, counted from 0.
    n : int
        How many strategies the player has, i.e. the length of the vector.

    Returns
    -------
    np.ndarray of length n, with 1 in position k and 0 elsewhere.

    Examples
    --------
    >>> pure(0, 2)    # "always Cooperate" in the prisoner's dilemma
    array([1., 0.])
    >>> pure(2, 3)    # "always Scissors" in rock-paper-scissors
    array([0., 0., 1.])
    '''
    v = np.zeros(n)
    v[k] = 1.0
    return v

In [3]:
show_game("prisoners_dilemma")

,Cooperate,Defect
Cooperate,"3, 3","0, 5"
Defect,"5, 0","1, 1"


Reading one cell back from the matrices: `A[1, 0] = 5` and `B[1, 0] = 0`. If the row player defects while the column player cooperates, the defector gets 5 and the cooperator gets nothing.

Bach or Stravinsky is the name Osborne & Rubinstein (1994) give to a game first described by Luce & Raiffa (1957), where it appears as the "battle of the sexes": two friends want to go to a concert together but prefer different composers. Two of these games are **zero-sum**: in every cell the payoffs add up to zero, so $B = -A$. Whatever one player gains, the other loses. The prisoner's dilemma is not zero-sum, since both players can do well at the same time in (Cooperate, Cooperate).

In [4]:
for name, (A, B) in GAMES.items():
    print(f"{name:22s} zero-sum: {np.array_equal(B, -A)}")

prisoners_dilemma      zero-sum: False
matching_pennies       zero-sum: True
rock_paper_scissors    zero-sum: True
bach_or_stravinsky     zero-sum: False
coordination           zero-sum: False
dominance_solvable     zero-sum: False


## 2 · Expected payoffs

When both players randomise independently, the probability of landing in cell $(i, j)$ is $x_i y_j$. The expected payoff of a player with payoff matrix $M$ is therefore

$$u(x, y) \;=\; \sum_i \sum_j x_i\, y_j\, M_{ij} \;=\; x^\top M y .$$

In NumPy this is a single line. The order of the product matters: $x$ indexes rows, so it multiplies from the left, and $y$ indexes columns, so it multiplies from the right. Written the other way around the dimensions stop matching as soon as the game is not square.

It helps to read the product in two steps. $M y$ is a vector with one entry per row, namely the payoff of that row against $y$. Multiplying by $x$ then averages those entries with the row player's own probabilities. The first step, on its own, is exactly what the next section needs.

In [5]:
def expected_payoff(M, x, y):
    '''Expected payoff x^T M y when both players randomise independently.

    Parameters
    ----------
    M : np.ndarray, shape (m, n)
        Payoff matrix of the player whose payoff we want (A or B).
    x : np.ndarray, shape (m,)
        Row player's mixed strategy.
    y : np.ndarray, shape (n,)
        Column player's mixed strategy.

    Returns
    -------
    float
    '''
    return float(x @ M @ y)

In [6]:
A, B = GAMES["prisoners_dilemma"]
x = np.array([0.5, 0.5])        # row player flips a coin
y = pure(0, 2)                  # column player always cooperates

print("payoff of each row against y:", A @ y)
print("row player's expected payoff:", expected_payoff(A, x, y))

payoff of each row against y: [3. 5.]
row player's expected payoff: 4.0


In [7]:
# sanity checks
A, B = GAMES["rock_paper_scissors"]
u = np.ones(3) / 3
assert abs(expected_payoff(A, u, u)) < TOL
A, B = GAMES["prisoners_dilemma"]
assert expected_payoff(A, pure(1, 2), pure(0, 2)) == 5
assert expected_payoff(B, pure(1, 2), pure(0, 2)) == 0
print("✓ expected_payoff")

✓ expected_payoff


## 3 · Best responses

A strategy is a best response to $y$ if no other strategy earns more against $y$. Among pure strategies this is simply the set of rows that attain $\max_i (M y)_i$.

Two details are easy to get wrong.

**Ties.** The best response need not be unique, and ties are not a corner case we can ignore. They are the heart of mixed equilibria: a player is only willing to randomise between strategies that do equally well. The function therefore returns *all* maximising rows.

**Floating point.** Two rows that are equal on paper may come out as `0.30000000000000004` and `0.3` after a few multiplications. Comparing with `==` would silently drop one of them. We instead accept every row within `TOL` of the maximum. Only the lower side needs slack, since nothing can exceed the maximum.

In [8]:
def best_response(M, y):
    '''Pure best responses of a row player against the opponent's mix y.

    Parameters
    ----------
    M : np.ndarray, shape (m, n)
        The responding player's payoffs, with her strategies as rows.
        Use A for the row player and B.T for the column player.
    y : np.ndarray, shape (n,)
        The opponent's mixed strategy.

    Returns
    -------
    np.ndarray of int
        Every row within TOL of the best achievable payoff (ties included).
    '''
    vals = M @ y
    return np.flatnonzero(vals >= vals.max() - TOL)

In [9]:
A, B = GAMES["prisoners_dilemma"]
print("row BR to Cooperate:", best_response(A, pure(0, 2)))
print("row BR to Defect:   ", best_response(A, pure(1, 2)))

A, B = GAMES["matching_pennies"]
print("row BR to a fair coin:", best_response(A, np.array([0.5, 0.5])))

row BR to Cooperate: [1]
row BR to Defect:    [1]
row BR to a fair coin: [0 1]


Defect is the best reply to everything in the prisoner's dilemma, which is what makes the game a dilemma. In matching pennies, a fair coin leaves the opponent completely indifferent; both of her strategies are best responses.

In [10]:
# sanity checks
A, B = GAMES["prisoners_dilemma"]
assert list(best_response(A, pure(0, 2))) == [1]
assert list(best_response(A, pure(1, 2))) == [1]
assert list(best_response(B.T, pure(0, 2))) == [1]     # column player via B.T
A, B = GAMES["matching_pennies"]
assert list(best_response(A, np.array([0.5, 0.5]))) == [0, 1]
print("✓ best_response")

✓ best_response


## 4 · Verifying an equilibrium

A profile $(x, y)$ is a Nash equilibrium when neither player can gain by deviating unilaterally:

$$x^\top A y \;\ge\; x'^\top A y \quad \forall x' \in \Delta^m, \qquad x^\top B y \;\ge\; x^\top B y' \quad \forall y' \in \Delta^n .$$

At first sight this asks us to check infinitely many deviations. It does not. The payoff of any mixed deviation $x'$ is a weighted average of the payoffs of pure deviations, and an average can never beat its largest term. So it is enough to compare the current payoff with the best **pure** deviation, i.e. with $\max_i (A y)_i$.

The equilibrium concept itself is from Nash (1950, 1951). The 1950 note proves existence in one page with Kakutani's fixed-point theorem; the 1951 paper uses Brouwer's. Both are short and worth reading in the original.

This small function will do a lot of work later. Support enumeration generates candidate profiles and uses `is_nash` to throw away the ones that fail, and in the puzzle game it is the check behind "have you found the equilibrium?".

In [11]:
def is_nash(A, B, x, y):
    '''Check whether (x, y) is a Nash equilibrium of the game (A, B).

    Only pure deviations are tested, which is sufficient (see the text).

    Parameters
    ----------
    A, B : np.ndarray, shape (m, n)
        Payoff matrices of the row and the column player.
    x : np.ndarray, shape (m,)
        Row player's mixed strategy.
    y : np.ndarray, shape (n,)
        Column player's mixed strategy.

    Returns
    -------
    bool
    '''
    row_ok = (A @ y).max() <= x @ A @ y + TOL
    col_ok = (x @ B).max() <= x @ B @ y + TOL
    return bool(row_ok and col_ok)

In [12]:
A, B = GAMES["prisoners_dilemma"]
print("(Defect, Defect):      ", is_nash(A, B, pure(1, 2), pure(1, 2)))
print("(Cooperate, Cooperate):", is_nash(A, B, pure(0, 2), pure(0, 2)))

A, B = GAMES["matching_pennies"]
h = np.array([0.5, 0.5])
print("fair coins:            ", is_nash(A, B, h, h))
print("biased row coin:       ", is_nash(A, B, np.array([0.6, 0.4]), h))

(Defect, Defect):       True
(Cooperate, Cooperate): False
fair coins:             True
biased row coin:        False


The last line is worth a moment. If the row player's coin leans towards Heads, the column player (who wants to mismatch) should switch to Tails for sure. Being even slightly predictable is exploitable, which is why the fair coin is the only equilibrium of this game.

In [13]:
# sanity checks
A, B = GAMES["prisoners_dilemma"]
assert is_nash(A, B, pure(1, 2), pure(1, 2))
assert not is_nash(A, B, pure(0, 2), pure(0, 2))
A, B = GAMES["matching_pennies"]
assert is_nash(A, B, h, h) and not is_nash(A, B, np.array([0.6, 0.4]), h)
print("✓ is_nash")

✓ is_nash


## 5 · Pure-strategy equilibria

A cell $(i, j)$ is a pure equilibrium exactly when $i$ is a best response to $j$ and $j$ is a best response to $i$. Checking every cell is the familiar "underline the best responses" method, done by a loop instead of a pencil.

The cost is $m \cdot n$ best-response computations, which is cheap. The catch is that many games have no pure equilibrium at all.

In [14]:
def pure_nash(A, B):
    '''All pure-strategy Nash equilibria of the game (A, B).

    Returns
    -------
    list of (i, j)
        Row index i and column index j of each equilibrium cell, in row order.
    '''
    m, n = A.shape
    out = []
    for i in range(m):
        for j in range(n):
            row_best = i in best_response(A, pure(j, n))
            col_best = j in best_response(B.T, pure(i, m))
            if row_best and col_best:
                out.append((i, j))
    return out

In [15]:
for name in GAMES:
    print(f"{name:22s} {pure_nash(*GAMES[name])}")

prisoners_dilemma      [(1, 1)]
matching_pennies       []
rock_paper_scissors    []
bach_or_stravinsky     [(0, 0), (1, 1)]
coordination           [(0, 0), (1, 1)]
dominance_solvable     [(0, 0)]


Three patterns show up. The prisoner's dilemma has a single pure equilibrium; Bach or Stravinsky and the coordination game have two, so a player cannot predict the outcome from the equilibrium concept alone; matching pennies and rock-paper-scissors have none. Nash's theorem guarantees an equilibrium in mixed strategies for every finite game, so for the last two we will have to look further (sections 7 and 8).

In [16]:
# sanity checks
assert pure_nash(*GAMES["prisoners_dilemma"]) == [(1, 1)]
assert pure_nash(*GAMES["bach_or_stravinsky"]) == [(0, 0), (1, 1)]
assert pure_nash(*GAMES["matching_pennies"]) == []
assert pure_nash(*GAMES["rock_paper_scissors"]) == []
print("✓ pure_nash")

✓ pure_nash


## 6 · Iterated elimination of strictly dominated strategies

Row $k$ **strictly dominates** row $i$ if it pays more in every column: $A_{kj} > A_{ij}$ for all $j$. A rational player never plays a strictly dominated strategy, whatever she believes about her opponent, so we may delete it.

Deletion changes the game. Once a row is gone, a column that used to be worthwhile only against that row may become dominated in turn. We therefore alternate between the players until nothing more can be removed. For strict dominance the order of deletions does not affect the final result (§3.4.3), which is what makes the procedure well defined. For weak dominance it does, and questions such as "can this strategy be eliminated by *some* order?" become computationally hard (Gilboa, Kalai & Zemel, 1993).

Two remarks on the implementation. We keep lists of the *original* indices that survive, so the answer can be read against the full game. And we record each deletion, because the trace is often more instructive than the final answer.

In [17]:
def _dominated_row(M):
    '''Find one row of M strictly dominated by another pure row.

    Returns
    -------
    (i, k) or None
        i is dominated by k (indices within M), or None if no row is dominated.
    '''
    for i in range(M.shape[0]):
        for k in range(M.shape[0]):
            if k != i and np.all(M[k] > M[i]):
                return i, k
    return None


def eliminate_dominated(A, B, trace=False):
    '''Iterated elimination of strategies strictly dominated by pure strategies.

    Parameters
    ----------
    A, B : np.ndarray, shape (m, n)
        Payoff matrices of the row and the column player.
    trace : bool
        If True, also return the list of deletions in the order they happened.

    Returns
    -------
    rows, cols : list of int
        Original indices of the surviving strategies.
    log : list of (side, removed, dominated_by), only if trace=True
    '''
    rows = list(range(A.shape[0]))
    cols = list(range(A.shape[1]))
    log = []
    changed = True
    while changed:
        changed = False
        hit = _dominated_row(A[np.ix_(rows, cols)])
        if hit is not None:
            i, k = hit
            log.append(("row", rows[i], rows[k]))
            rows.pop(i)
            changed = True
        hit = _dominated_row(B[np.ix_(rows, cols)].T)
        if hit is not None:
            j, k = hit
            log.append(("col", cols[j], cols[k]))
            cols.pop(j)
            changed = True
    return (rows, cols, log) if trace else (rows, cols)

In [18]:
show_game("dominance_solvable")

,L,C,R
U,"4, 3","5, 1","6, 2"
M,"2, 1","8, 4","3, 6"
D,"3, 0","9, 6","2, 8"


In [19]:
A, B = GAMES["dominance_solvable"]
rlab, clab = LABELS["dominance_solvable"]
rows, cols, log = eliminate_dominated(A, B, trace=True)

for side, removed, by in log:
    lab = rlab if side == "row" else clab
    print(f"{side:3s} {lab[removed]} is dominated by {lab[by]}  → removed")
print("survivors:", [rlab[i] for i in rows], [clab[j] for j in cols])

col C is dominated by R  → removed
row M is dominated by U  → removed
row D is dominated by U  → removed
col R is dominated by L  → removed
survivors: ['U'] ['L']


Follow the trace against the table above. At the start no row is dominated: each of U, M and D is best somewhere. The column player, however, never wants C, since R pays her more in every row (2 > 1, 6 > 4, 8 > 6). Once C is gone, the row player's comparison changes. Without the column where M and D looked attractive, U beats both of them. Finally, facing only U, the column player prefers L (3) to R (2), and a single cell is left. That cell is also the unique Nash equilibrium of the game. This is a general fact: if iterated strict dominance leaves a single profile, that profile is the unique equilibrium.

Note that we only tested domination by **pure** strategies here. A strategy can survive this test and still be dominated by a mixture of other strategies. Section 9 handles that case with an LP.

In [20]:
# sanity checks
assert eliminate_dominated(*GAMES["prisoners_dilemma"]) == ([1], [1])
r, c = eliminate_dominated(*GAMES["dominance_solvable"])
assert len(r) == 1 and len(c) == 1
assert (r[0], c[0]) in pure_nash(*GAMES["dominance_solvable"])
assert eliminate_dominated(*GAMES["matching_pennies"]) == ([0, 1], [0, 1])
print("✓ eliminate_dominated")

✓ eliminate_dominated


## 7 · Zero-sum games as linear programs

In a zero-sum game the interests are perfectly opposed, and this lets us assume the worst about the opponent without losing anything. The row player picks the mixed strategy whose *guaranteed* payoff is highest:

$$\max_{x \in \Delta^m} \; \min_{j} \; (x^\top A)_j .$$

The inner minimum is not linear, but the standard epigraph trick fixes that. Introduce a variable $v$ for the guaranteed payoff and require it to lie below every column's payoff:

$$\begin{aligned}
\max_{x,\,v} \quad & v \\
\text{s.t.} \quad & \textstyle\sum_i x_i A_{ij} \;\ge\; v \qquad j = 1, \dots, n \\
& \textstyle\sum_i x_i = 1, \qquad x \ge 0, \qquad v \text{ free.}
\end{aligned}$$

By von Neumann's (1928) minimax theorem the optimal $v$ equals what the column player can hold the row player down to, and the pair of optimal strategies is a Nash equilibrium (§3.4.1). The optimal $v$ is called the **value** of the game.

Translating to `scipy.optimize.linprog` needs a little bookkeeping:

- `linprog` minimises, so the objective is $-v$;
- the decision vector is $z = (x_1, \dots, x_m, v)$;
- constraints must read $A_{\text{ub}}\, z \le b_{\text{ub}}$, so each $\sum_i x_i A_{ij} \ge v$ becomes $-\sum_i x_i A_{ij} + v \le 0$.

The link runs both ways. Dantzig (1951) showed that every zero-sum game is an LP, and conversely that LPs can be written as zero-sum games; Adler (2013) closes a gap in the converse direction and is a clear modern account. The column player's problem is exactly the LP dual of the row player's, so LP duality *is* the minimax theorem in disguise.

In [21]:
def solve_zero_sum(A):
    '''Maxmin strategy and value of the zero-sum game (A, -A), via an LP.

    Parameters
    ----------
    A : np.ndarray, shape (m, n)
        Row player's payoffs. The column player's payoffs are -A.

    Returns
    -------
    x : np.ndarray, shape (m,)
        Row player's optimal mixed strategy.
    v : float
        Value of the game for the row player.
    '''
    m, n = A.shape
    c = np.zeros(m + 1)
    c[-1] = -1                                      # maximise v

    A_ub = np.hstack([-A.T, np.ones((n, 1))])       # -x·A[:, j] + v <= 0
    b_ub = np.zeros(n)
    A_eq = np.hstack([np.ones((1, m)), np.zeros((1, 1))])
    bounds = [(0, None)] * m + [(None, None)]

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1], bounds=bounds)
    return res.x[:m], res.x[-1]

In [22]:
for name in ["matching_pennies", "rock_paper_scissors"]:
    x, v = solve_zero_sum(GAMES[name][0])
    print(f"{name:20s} x = {x}, value = {round(v, 10) + 0.0:.4f}")

A = np.array([[3, -1], [-2, 1]])
x, v = solve_zero_sum(A)
y, _ = solve_zero_sum(-A.T)        # the column player's problem, via the B.T trick
print(f"{'asymmetric 2x2':20s} x = {x}, y = {y}, value = {v:.4f}")

matching_pennies     x = [0.5 0.5], value = 0.0000
rock_paper_scissors  x = [0.3333 0.3333 0.3333], value = 0.0000
asymmetric 2x2       x = [0.4286 0.5714], y = [0.2857 0.7143], value = 0.1429


The last game is less symmetric and makes a good hand check. Let the row player play the top row with probability $p$. Against the left column she earns $3p - 2(1-p) = 5p - 2$; against the right column, $-p + (1-p) = 1 - 2p$. The guaranteed payoff is the lower of the two lines, and it is highest where they cross: $p = 3/7$, giving a value of $1/7$. The LP finds the same point.

Also note how the column player's strategy was obtained. In $-A^\top$ the column player becomes a row player maximising her own payoff, so the same function solves her problem.

> **Going further.** Instead of solving a second LP, read $y$ off the dual variables of the first one (`res.ineqlin.marginals` in SciPy). Verifying that they agree is a good way to see LP duality and the minimax theorem as the same statement.

In [23]:
# sanity checks
x, v = solve_zero_sum(GAMES["matching_pennies"][0])
assert np.allclose(x, [0.5, 0.5]) and abs(v) < 1e-7
x, v = solve_zero_sum(GAMES["rock_paper_scissors"][0])
assert np.allclose(x, [1/3, 1/3, 1/3]) and abs(v) < 1e-7
A = np.array([[3, -1], [-2, 1]])
x, v = solve_zero_sum(A)
y, _ = solve_zero_sum(-A.T)
assert np.allclose(x, [3/7, 4/7]) and abs(v - 1/7) < 1e-7
assert is_nash(A, -A, x, y)
print("✓ solve_zero_sum")

✓ solve_zero_sum


## 8 · Support enumeration

Outside zero-sum games the maxmin argument breaks down: the opponent is no longer trying to hurt you, so planning for the worst case is not the same as playing an equilibrium. We need a different idea.

The **support** of a mixed strategy is the set of pure strategies it plays with positive probability. The key observation is:

> In equilibrium, every strategy in a player's support earns the same expected payoff, and no strategy outside the support earns more.

If one strategy in the support did worse, the player would move its probability to a better one. So *if we knew the supports*, finding the equilibrium would reduce to solving linear equations. We do not know them, so we try all of them. This is the algorithm of §4.2.3; von Stengel (2002, §2) gives a cleaner treatment together with the precise definition of nondegeneracy used below.

### 8a · The indifference system

Fix a row support $R$ and a column support $C$ of equal size $k$. We look for $y$, supported on $C$, that makes the row player indifferent across $R$. The unknowns are the $k$ probabilities $y_j$, $j \in C$, and the common payoff $u$:

$$\sum_{j \in C} A_{ij}\, y_j \;=\; u \quad \forall i \in R, \qquad \sum_{j \in C} y_j = 1 .$$

That is $k+1$ equations in $k+1$ unknowns. If the system is singular we give up on this pair of supports and return `None`.

This is the part that feels backwards at first: **$y$ is pinned down by the row player's payoffs, not the column player's own.** Each player mixes so as to keep the *other* player willing to mix. The row player's $x$ is found the same way, by making the column player indifferent, which in code is `solve_indifference(B.T, C, R)`.

In [24]:
def solve_indifference(M, rows, cols):
    '''Opponent mix on `cols` that makes this player indifferent over `rows`.

    Parameters
    ----------
    M : np.ndarray, shape (m, n)
        Payoffs of the player who must be made indifferent, strategies as rows.
    rows : sequence of int, length k
        Her support: the strategies that must all earn the same payoff.
    cols : sequence of int, length k
        The opponent's support: the only strategies allowed positive probability.

    Returns
    -------
    np.ndarray of length n, or None
        The opponent's mix (zeros outside `cols`), or None if the system is singular.
        Probabilities may be negative; the caller must check.
    '''
    k = len(rows)
    S = np.zeros((k + 1, k + 1))
    rhs = np.zeros(k + 1)
    S[:k, :k] = M[np.ix_(rows, cols)]   # sum_j M[i, j] y_j ...
    S[:k, k] = -1                       # ... - u = 0   for every i in rows
    S[k, :k] = 1                        # sum_j y_j = 1
    rhs[k] = 1
    try:
        sol = np.linalg.solve(S, rhs)
    except np.linalg.LinAlgError:
        return None
    y = np.zeros(M.shape[1])
    y[list(cols)] = sol[:k]
    return y

In [25]:
A, B = GAMES["bach_or_stravinsky"]
y = solve_indifference(A, [0, 1], [0, 1])
x = solve_indifference(B.T, [0, 1], [0, 1])
print("x =", x, " y =", y)
print("row payoff of each row against y:   ", A @ y)
print("column payoff of each column vs x:  ", x @ B)

x = [0.6667 0.3333]  y = [0.3333 0.6667]
row payoff of each row against y:    [0.6667 0.6667]
column payoff of each column vs x:   [0.6667 0.6667]


The row player prefers the opera, yet in the mixed equilibrium she goes to the opera with probability $2/3$ only because that is what keeps the *column* player indifferent. Both of her rows earn exactly $2/3$ against $y$, and both of his columns earn $2/3$ against $x$. The indifference conditions hold by construction.

### 8b · Enumerating supports

For every size $k$ and every pair of supports of that size:

1. solve the two indifference systems;
2. discard the pair if either system is singular or produces a negative probability;
3. keep the profile if `is_nash` accepts it. This last step catches the one condition we have not imposed yet, that no strategy *outside* the support does better.

Restricting to supports of equal size is justified for **nondegenerate** games, where in every equilibrium the two supports have the same size. Games with generic real payoffs are nondegenerate. Section 10 shows what goes wrong without this assumption.

Brute force is not the only way to search this space. Porter, Nudelman & Shoham (2008) enumerate supports from small to large and prune dominated strategies along the way; on many benchmark games this finds *a* first equilibrium much faster than smarter-looking methods. The classical alternative is the Lemke–Howson algorithm (Lemke & Howson, 1964), which follows a path of almost-equilibria and always ends at an equilibrium.

In [26]:
def support_enumeration(A, B):
    '''All Nash equilibria of a nondegenerate bimatrix game.

    Parameters
    ----------
    A, B : np.ndarray, shape (m, n)
        Payoff matrices of the row and the column player.

    Returns
    -------
    list of (x, y)
        Each equilibrium as a pair of mixed strategies.
    '''
    m, n = A.shape
    eqs = []
    for k in range(1, min(m, n) + 1):
        for rows in itertools.combinations(range(m), k):
            for cols in itertools.combinations(range(n), k):
                y = solve_indifference(A, rows, cols)
                x = solve_indifference(B.T, cols, rows)
                if x is None or y is None:
                    continue
                if (x < -TOL).any() or (y < -TOL).any():
                    continue
                x, y = np.clip(x, 0, None), np.clip(y, 0, None)
                if is_nash(A, B, x, y):
                    eqs.append((x, y))
    return eqs

In [27]:
for name in ["bach_or_stravinsky", "rock_paper_scissors", "prisoners_dilemma"]:
    print(name)
    for x, y in support_enumeration(*GAMES[name]):
        A, B = GAMES[name]
        print(f"   x = {x}, y = {y}, payoffs = ({expected_payoff(A, x, y):.3f}, {expected_payoff(B, x, y):.3f})")

bach_or_stravinsky
   x = [1. 0.], y = [1. 0.], payoffs = (2.000, 1.000)
   x = [0. 1.], y = [0. 1.], payoffs = (1.000, 2.000)
   x = [0.6667 0.3333], y = [0.3333 0.6667], payoffs = (0.667, 0.667)
rock_paper_scissors
   x = [0.3333 0.3333 0.3333], y = [0.3333 0.3333 0.3333], payoffs = (0.000, 0.000)
prisoners_dilemma
   x = [0. 1.], y = [0. 1.], payoffs = (1.000, 1.000)


Bach or Stravinsky now shows all three equilibria: the two pure ones we already had and the mixed one from 8a. Notice the payoffs of the mixed equilibrium. Each player gets $2/3$, less than the worse of the two pure equilibria. Randomising independently means the couple ends up in different places $5/9$ of the time, and that miscoordination is costly for both.

As a final check we compare against `nashpy` (Knight & Campbell, 2018), an independent implementation, on 30 random games with real-valued payoffs.

In [28]:
def same_equilibria(E1, E2):
    '''True if two lists of equilibria coincide up to order and rounding.'''
    if len(E1) != len(E2):
        return False
    used = set()
    for x1, y1 in E1:
        match = next((k for k, (x2, y2) in enumerate(E2)
                      if k not in used
                      and np.allclose(x1, x2, atol=1e-6)
                      and np.allclose(y1, y2, atol=1e-6)), None)
        if match is None:
            return False
        used.add(match)
    return True


# sanity checks
assert len(support_enumeration(*GAMES["bach_or_stravinsky"])) == 3
E = support_enumeration(*GAMES["rock_paper_scissors"])
assert len(E) == 1 and np.allclose(E[0][0], 1/3)

rng = np.random.default_rng(0)
for trial in range(30):
    m, n = rng.integers(2, 5, size=2)
    A, B = rng.random((m, n)), rng.random((m, n))
    assert same_equilibria(support_enumeration(A, B),
                           list(nash.Game(A, B).support_enumeration())), f"mismatch in trial {trial}"
print("✓ support_enumeration agrees with nashpy on 30 random games")

✓ support_enumeration agrees with nashpy on 30 random games


## 9 · Domination by mixed strategies

Consider a row player with three rows paying $(3, 0)$, $(1, 1)$ and $(0, 3)$ against two columns. Neither the top nor the bottom row beats the middle one in both columns, so the test of section 6 keeps it. Yet the half-half mixture of top and bottom pays $(1.5, 1.5)$ and beats the middle row everywhere. The middle row should never be played.

Whether row $i$ is dominated by *some* mixture of the other rows is again an LP. We search for the mixture $p$ with the largest uniform margin $\varepsilon$ over row $i$:

$$\max_{p,\,\varepsilon} \; \varepsilon \quad \text{s.t.} \quad \sum_{k \ne i} p_k M_{kj} \;\ge\; M_{ij} + \varepsilon \;\; \forall j, \qquad \sum_{k \ne i} p_k = 1, \qquad p \ge 0 .$$

Row $i$ is strictly dominated exactly when the optimal $\varepsilon$ is positive. The book (§4.5.2) uses an equivalent formulation that avoids the extra variable by assuming positive payoffs and normalising; comparing the two is a nice exercise in LP modelling.

This test matters more than it looks. In two-player games, a strategy is strictly dominated by a mixed strategy **if and only if** it is never a best response to any belief about the opponent (Pearce, 1984). So the LP above is also a test of whether a strategy is rationalizable in one round (Bernheim, 1984; Pearce, 1984).

In [29]:
def dominated_by_mixed(M, i):
    '''Is row i of M strictly dominated by some mixture of the other rows?

    Parameters
    ----------
    M : np.ndarray, shape (m, n)
        Payoffs of the player in question, strategies as rows.
    i : int
        The row to test.

    Returns
    -------
    bool
    '''
    others = [k for k in range(M.shape[0]) if k != i]
    q, n = len(others), M.shape[1]
    c = np.zeros(q + 1)
    c[-1] = -1                                              # maximise epsilon
    A_ub = np.hstack([-M[others].T, np.ones((n, 1))])       # -p·M[:, j] + eps <= -M[i, j]
    b_ub = -M[i]
    A_eq = np.hstack([np.ones((1, q)), np.zeros((1, 1))])
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=[1],
                  bounds=[(0, None)] * q + [(None, None)])
    return bool(res.x[-1] > TOL)

In [30]:
M = np.array([[3, 0], [1, 1], [0, 3]])
print("dominated by a pure row?  ", _dominated_row(M) is not None)
print("row 1 dominated by a mix? ", dominated_by_mixed(M, 1))
print("row 0 dominated by a mix? ", dominated_by_mixed(M, 0))

dominated by a pure row?   False
row 1 dominated by a mix?  True
row 0 dominated by a mix?  False


In [31]:
# sanity checks
assert _dominated_row(M) is None
assert dominated_by_mixed(M, 1) and not dominated_by_mixed(M, 0)
print("✓ dominated_by_mixed")

✓ dominated_by_mixed


## 10 · Discussion

A few questions worth thinking through once the code works.

**Why does the complexity of support enumeration matter?**
The number of support pairs of equal size is

$$\sum_{k=1}^{\min(m,n)} \binom{m}{k}\binom{n}{k} \;=\; \binom{m+n}{n} - 1 ,$$

by Vandermonde's identity. This grows exponentially. For a $10 \times 10$ game it is already close to two hundred thousand linear systems (see the cell below). The method is perfectly fine for small games like the ones here and hopeless for large ones. §4.2.1 explains why we should not expect a polynomial algorithm: computing even one equilibrium is PPAD-complete, first for games with three or more players (Daskalakis, Goldberg & Papadimitriou, 2009) and then for two-player games as well (Chen, Deng & Teng, 2009).

In [32]:
for s in [2, 3, 5, 10, 15]:
    print(f"{s:2d} x {s:<2d}: {comb(2 * s, s) - 1:>12,d} support pairs")

 2 x 2 :            5 support pairs
 3 x 3 :           19 support pairs
 5 x 5 :          251 support pairs
10 x 10:      184,755 support pairs
15 x 15:  155,117,519 support pairs


**What happens in a degenerate game?**
Take the most degenerate game there is: every payoff is zero. Every profile, pure or mixed, is an equilibrium, since nobody can gain by deviating.

In [33]:
Z = np.zeros((2, 2))
E = support_enumeration(Z, Z)
print(f"support enumeration returns {len(E)} equilibria:")
for x, y in E:
    print("  ", x, y)

support enumeration returns 4 equilibria:
   [1. 0.] [1. 0.]
   [1. 0.] [0. 1.]
   [0. 1.] [1. 0.]
   [0. 1.] [0. 1.]


It returns the four pure profiles and misses the continuum of mixed ones. With $k = 2$ the indifference system is singular (every row is trivially indifferent, so the equations carry no information about $y$), and the algorithm skips it. In degenerate games equilibria can form continua and the two supports can differ in size, so the equal-size enumeration is no longer complete. Avis, Rosenberg, Savani & von Stengel (2010) describe how to enumerate all equilibria, including these components, by working with the vertices of the best-response polytopes instead. For the puzzle generator this has a practical consequence: we will only generate nondegenerate games, or check degeneracy explicitly.

**Why is it enough to check pure deviations?** Covered in section 4: a mixed deviation's payoff is an average of pure deviations' payoffs.

**Why does a player choose her mix to make the *other* player indifferent?** Covered in section 8a. Her own indifference is determined by the opponent's mix. Her own probabilities matter only through their effect on the opponent. If she leaned one way, the opponent would stop mixing and exploit her.

**Open question for the project.** Which of these games would be easy for a human and hard for a learning algorithm, or the other way round? Dominance-solvable games are easy for both. Large games with a single mixed equilibrium are trivial for the LP and very hard for a person. Rock-paper-scissors is easy to reason about, yet simple learning rules tend to circle around its equilibrium rather than settle on it. Fictitious play, the oldest such rule (Brown, 1951), is known to converge in zero-sum games (Robinson, 1951) but can cycle forever in general games; Shapley (1964) gave a 3×3 example. Learning dynamics are the next step of the project.

---
## References

**Textbooks**

- Osborne, M. J. (2004). *An Introduction to Game Theory*. Oxford University Press.
- Osborne, M. J., & Rubinstein, A. (1994). *A Course in Game Theory*. MIT Press.
- Shoham, Y., & Leyton-Brown, K. (2009). *Multiagent Systems: Algorithmic, Game-Theoretic, and Logical Foundations*. Cambridge University Press.
- von Neumann, J., & Morgenstern, O. (1944). *Theory of Games and Economic Behavior*. Princeton University Press.
- Luce, R. D., & Raiffa, H. (1957). *Games and Decisions*. Wiley.

**Equilibrium and minimax**

- Nash, J. F. (1950). Equilibrium points in n-person games. *Proceedings of the National Academy of Sciences*, 36(1), 48–49.
- Nash, J. F. (1951). Non-cooperative games. *Annals of Mathematics*, 54(2), 286–295.
- von Neumann, J. (1928). Zur Theorie der Gesellschaftsspiele. *Mathematische Annalen*, 100, 295–320.
- Dantzig, G. B. (1951). A proof of the equivalence of the programming problem and the game problem. In T. C. Koopmans (Ed.), *Activity Analysis of Production and Allocation* (pp. 330–335). Wiley.
- Adler, I. (2013). The equivalence of linear programs and zero-sum games. *International Journal of Game Theory*, 42(1), 165–177.

**Dominance and rationalizability**

- Bernheim, B. D. (1984). Rationalizable strategic behavior. *Econometrica*, 52(4), 1007–1028.
- Pearce, D. G. (1984). Rationalizable strategic behavior and the problem of perfection. *Econometrica*, 52(4), 1029–1050.
- Gilboa, I., Kalai, E., & Zemel, E. (1993). The complexity of eliminating dominated strategies. *Mathematics of Operations Research*, 18(3), 553–565.

**Computing equilibria**

- Lemke, C. E., & Howson, J. T. (1964). Equilibrium points of bimatrix games. *Journal of the Society for Industrial and Applied Mathematics*, 12(2), 413–423.
- von Stengel, B. (2002). Computing equilibria for two-person games. In R. J. Aumann & S. Hart (Eds.), *Handbook of Game Theory with Economic Applications* (Vol. 3, pp. 1723–1759). Elsevier.
- Porter, R., Nudelman, E., & Shoham, Y. (2008). Simple search methods for finding a Nash equilibrium. *Games and Economic Behavior*, 63(2), 642–662.
- Avis, D., Rosenberg, G. D., Savani, R., & von Stengel, B. (2010). Enumeration of Nash equilibria for two-player games. *Economic Theory*, 42(1), 9–37.
- Daskalakis, C., Goldberg, P. W., & Papadimitriou, C. H. (2009). The complexity of computing a Nash equilibrium. *SIAM Journal on Computing*, 39(1), 195–259.
- Chen, X., Deng, X., & Teng, S.-H. (2009). Settling the complexity of computing two-player Nash equilibria. *Journal of the ACM*, 56(3), 1–57.
- Knight, V., & Campbell, J. (2018). Nashpy: A Python library for the computation of Nash equilibria. *Journal of Open Source Software*, 3(30), 904.

**Learning in games (next notebook)**

- Brown, G. W. (1951). Iterative solution of games by fictitious play. In T. C. Koopmans (Ed.), *Activity Analysis of Production and Allocation* (pp. 374–376). Wiley.
- Robinson, J. (1951). An iterative method of solving a game. *Annals of Mathematics*, 54(2), 296–301.
- Shapley, L. S. (1964). Some topics in two-person games. In M. Dresher, L. S. Shapley, & A. W. Tucker (Eds.), *Advances in Game Theory* (pp. 1–28). Princeton University Press.

---
### Where these functions go next

These functions become `engine/solvers.py`, and the sanity checks become `tests/test_solvers.py` for pytest. The next notebook builds learning agents (fictitious play, regret matching, Q-learning) on top of them.